# Component 1 — B1 and B2 Aspect Detection Baselines

In [8]:

from pathlib import Path
import json, random, time
import numpy as np
import pandas as pd
import torch
import yaml

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import precision_recall_fscore_support, f1_score, accuracy_score, hamming_loss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from IPython.display import display

CURRENT = Path.cwd().resolve()
if CURRENT.name == "notebooks":
    PROJECT_ROOT = CURRENT.parent
elif (CURRENT / "data").exists():
    PROJECT_ROOT = CURRENT
elif (CURRENT.parent / "data").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    raise RuntimeError(f"Could not locate component_1 from {CURRENT}")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "mabsa_multilabel_reviews.csv"
TAXONOMY_PATH = PROJECT_ROOT / "taxonomy" / "aspect_taxonomy_v0_2.yaml"
RESULTS_DIR = PROJECT_ROOT / "results" / "baselines"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
MODEL_NAME = "roberta-base"
MAX_LENGTH = 192
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRAD_CLIP_NORM = 1.0
SMOKE_TEST = False
SAVE_MODEL_WEIGHTS = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
if DEVICE.type == "cpu":
    print("WARNING: roberta-base training on CPU can be slow.")


Device: cpu
Model: roberta-base


In [9]:

for path in [DATA_PATH, TAXONOMY_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

data = pd.read_csv(DATA_PATH)
data["canonical_aspects"] = data["canonical_aspects"].apply(json.loads)
data["semantic_groups"] = data["semantic_groups"].apply(json.loads)

with open(TAXONOMY_PATH, "r", encoding="utf-8") as f:
    taxonomy = yaml.safe_load(f)

COMMON = taxonomy["groups"]["COMMON"]
PHONE_SPECIFIC = taxonomy["groups"]["PHONE_SPECIFIC"]
LAPTOP_SPECIFIC = taxonomy["groups"]["LAPTOP_SPECIFIC"]

GLOBAL_LABELS = COMMON + PHONE_SPECIFIC + LAPTOP_SPECIFIC
DOMAIN_LABELS = {
    "phone": COMMON + PHONE_SPECIFIC,
    "laptop": COMMON + LAPTOP_SPECIFIC,
}

if len(GLOBAL_LABELS) != len(set(GLOBAL_LABELS)):
    raise ValueError("Duplicate labels found in global label space.")

display(data.groupby(["domain", "split"]).size().rename("review_count").reset_index())
print("GLOBAL LABEL COUNT:", len(GLOBAL_LABELS))


,domain,split,review_count
0,laptop,dev,218
1,laptop,test,350
2,laptop,train,840
3,phone,dev,116
4,phone,test,223
5,phone,train,533


GLOBAL LABEL COUNT: 27


In [10]:

def split_data(df, domain=None):
    d = df.copy() if domain is None else df[df["domain"].eq(domain)].copy()
    train_df = d[d["split"].eq("train")].reset_index(drop=True)
    dev_df = d[d["split"].eq("dev")].reset_index(drop=True)
    test_df = d[d["split"].eq("test")].reset_index(drop=True)

    if SMOKE_TEST:
        train_df = train_df.head(128)
        dev_df = dev_df.head(64)
        test_df = test_df.head(64)

    return train_df, dev_df, test_df


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class MultiLabelDataset(Dataset):
    def __init__(self, dataframe, label_names):
        self.df = dataframe.reset_index(drop=True)
        self.label_names = label_names
        label_to_idx = {label: i for i, label in enumerate(label_names)}

        self.enc = tokenizer(
            self.df["text"].astype(str).tolist(),
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

        self.labels = np.zeros((len(self.df), len(label_names)), dtype=np.float32)

        for row_idx, aspects in enumerate(self.df["canonical_aspects"]):
            for aspect in aspects:
                if aspect not in label_to_idx:
                    raise ValueError(f"Aspect {aspect} not in active label space.")
                self.labels[row_idx, label_to_idx[aspect]] = 1.0

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item


In [11]:

def make_loaders(train_df, dev_df, test_df, labels):
    train_ds = MultiLabelDataset(train_df, labels)
    dev_ds = MultiLabelDataset(dev_df, labels)
    test_ds = MultiLabelDataset(test_df, labels)

    return (
        DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True),
        DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False),
        DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False),
    )


def evaluate_loss(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            losses.append(model(**batch).loss.item())
    return float(np.mean(losses)) if losses else 0.0


def collect_probs(model, loader):
    model.eval()
    probs_list, true_list = [], []

    with torch.no_grad():
        for batch in loader:
            true_list.append(batch["labels"].numpy())
            inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
            probs = torch.sigmoid(model(**inputs).logits).cpu().numpy()
            probs_list.append(probs)

    return np.vstack(probs_list), np.vstack(true_list)


def tune_threshold(y_true, probs):
    rows = []
    for threshold in np.arange(0.10, 0.91, 0.05):
        pred = (probs >= threshold).astype(int)
        score = f1_score(y_true, pred, average="micro", zero_division=0)
        rows.append({"threshold": round(float(threshold), 2), "micro_f1": float(score)})

    table = pd.DataFrame(rows)
    best = table.sort_values(["micro_f1", "threshold"], ascending=[False, True]).iloc[0]
    return float(best["threshold"]), table


def metric_dict(y_true, probs, threshold):
    pred = (probs >= threshold).astype(int)

    p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(
        y_true, pred, average="micro", zero_division=0
    )
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true, pred, average="macro", zero_division=0
    )

    return {
        "precision_micro": float(p_micro),
        "recall_micro": float(r_micro),
        "micro_f1": float(f_micro),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "macro_f1": float(f_macro),
        "exact_match": float(accuracy_score(y_true, pred)),
        "hamming_loss": float(hamming_loss(y_true, pred)),
    }, pred


In [12]:

def train_roberta(run_name, train_df, dev_df, test_df, labels):
    train_loader, dev_loader, test_loader = make_loaders(train_df, dev_df, test_df, labels)

    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(labels),
        problem_type="multi_label_classification",
        label2id=label2id,
        id2label=id2label,
    ).to(DEVICE)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    total_steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps,
    )

    history = []
    start = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []

        for batch in tqdm(train_loader, desc=f"{run_name} epoch {epoch}/{EPOCHS}"):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
            scheduler.step()
            losses.append(out.loss.item())

        row = {
            "model": run_name,
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "dev_loss": evaluate_loss(model, dev_loader),
        }
        history.append(row)
        print(row)

    dev_probs, dev_true = collect_probs(model, dev_loader)
    threshold, threshold_table = tune_threshold(dev_true, dev_probs)

    test_probs, test_true = collect_probs(model, test_loader)
    metrics, test_pred = metric_dict(test_true, test_probs, threshold)

    if SAVE_MODEL_WEIGHTS:
        torch.save(model.state_dict(), RESULTS_DIR / f"{run_name}_state_dict.pt")

    return {
        "model": model,
        "threshold": threshold,
        "labels": labels,
        "test_probs": test_probs,
        "test_true": test_true,
        "test_pred": test_pred,
        "metrics": metrics,
        "history": pd.DataFrame(history),
        "threshold_table": threshold_table.assign(model=run_name),
        "training_seconds": time.time() - start,
    }


In [13]:

def prediction_frame(run_name, test_df, result):
    labels = result["labels"]
    rows = []

    for i, row in test_df.reset_index(drop=True).iterrows():
        true_aspects = [labels[j] for j in range(len(labels)) if result["test_true"][i, j] == 1]
        predicted_aspects = [labels[j] for j in range(len(labels)) if result["test_pred"][i, j] == 1]

        rows.append({
            "model": run_name,
            "domain": row["domain"],
            "review_id": row["review_id"],
            "text": row["text"],
            "true_aspects": json.dumps(true_aspects, ensure_ascii=False),
            "predicted_aspects": json.dumps(predicted_aspects, ensure_ascii=False),
            "threshold": result["threshold"],
        })

    return pd.DataFrame(rows)


def per_label_frame(run_name, domain, result):
    p, r, f1, support = precision_recall_fscore_support(
        result["test_true"],
        result["test_pred"],
        average=None,
        zero_division=0,
    )

    return pd.DataFrame({
        "model": run_name,
        "domain": domain,
        "canonical_aspect": result["labels"],
        "precision": p,
        "recall": r,
        "f1": f1,
        "support": support.astype(int),
    })


metric_rows, history_parts, prediction_parts, per_label_parts = [], [], [], []

for domain in ["phone", "laptop"]:
    train_df, dev_df, test_df = split_data(data, domain)
    result = train_roberta(
        f"b1_{domain}",
        train_df,
        dev_df,
        test_df,
        DOMAIN_LABELS[domain],
    )

    metric_rows.append({
        "model": "b1",
        "segment": domain,
        **result["metrics"],
        "threshold": result["threshold"],
        "training_seconds": result["training_seconds"],
    })

    history_parts.append(result["history"])
    prediction_parts.append(prediction_frame("b1", test_df, result))
    per_label_parts.append(per_label_frame("b1", domain, result))

    del result["model"]
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

train_all, dev_all, test_all = split_data(data, None)
b2 = train_roberta("b2_pooled", train_all, dev_all, test_all, GLOBAL_LABELS)

metric_rows.append({
    "model": "b2",
    "segment": "overall",
    **b2["metrics"],
    "threshold": b2["threshold"],
    "training_seconds": b2["training_seconds"],
})

history_parts.append(b2["history"])
prediction_parts.append(prediction_frame("b2", test_all, b2))
per_label_parts.append(per_label_frame("b2", "overall", b2))

for domain in ["phone", "laptop"]:
    idx = test_all["domain"].eq(domain).to_numpy()
    domain_metrics, _ = metric_dict(
        b2["test_true"][idx],
        b2["test_probs"][idx],
        b2["threshold"],
    )

    metric_rows.append({
        "model": "b2",
        "segment": domain,
        **domain_metrics,
        "threshold": b2["threshold"],
        "training_seconds": b2["training_seconds"],
    })

del b2["model"]
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 908.38it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
b1_phone epoch 1/3: 100%|██████████| 67/67 [12:17<00:00, 11.01s/it]


{'model': 'b1_phone', 'epoch': 1, 'train_loss': 0.4646902818288376, 'dev_loss': 0.28825403849283854}


b1_phone epoch 2/3: 100%|██████████| 67/67 [10:53<00:00,  9.75s/it]


{'model': 'b1_phone', 'epoch': 2, 'train_loss': 0.28215522241236557, 'dev_loss': 0.26470420360565183}


b1_phone epoch 3/3: 100%|██████████| 67/67 [10:24<00:00,  9.32s/it]


...

{'model': 'b2_pooled', 'epoch': 1, 'train_loss': 0.3241572970914286, 'dev_loss': 0.17610117189940952}


b2_pooled epoch 2/3: 100%|██████████| 172/172 [26:32<00:00,  9.26s/it]


{'model': 'b2_pooled', 'epoch': 2, 'train_loss': 0.16904285034641278, 'dev_loss': 0.1484705332134451}


b2_pooled epoch 3/3: 100%|██████████| 172/172 [26:24<00:00,  9.21s/it]


{'model': 'b2_pooled', 'epoch': 3, 'train_loss': 0.14757220321443193, 'dev_loss': 0.139153401411715}


In [14]:

metrics_df = pd.DataFrame(metric_rows)
history_df = pd.concat(history_parts, ignore_index=True)
predictions_df = pd.concat(prediction_parts, ignore_index=True)
per_label_df = pd.concat(per_label_parts, ignore_index=True)

comparison_rows = []

for domain in ["phone", "laptop"]:
    b1_row = metrics_df[(metrics_df["model"] == "b1") & (metrics_df["segment"] == domain)].iloc[0]
    b2_row = metrics_df[(metrics_df["model"] == "b2") & (metrics_df["segment"] == domain)].iloc[0]

    comparison_rows.append({
        "domain": domain,
        "b1_micro_f1": b1_row["micro_f1"],
        "b2_micro_f1": b2_row["micro_f1"],
        "delta_micro_f1_b2_minus_b1": b2_row["micro_f1"] - b1_row["micro_f1"],
        "b1_macro_f1": b1_row["macro_f1"],
        "b2_macro_f1": b2_row["macro_f1"],
        "delta_macro_f1_b2_minus_b1": b2_row["macro_f1"] - b1_row["macro_f1"],
    })

comparison_df = pd.DataFrame(comparison_rows)

display(metrics_df)
display(comparison_df)


,model,segment,precision_micro,recall_micro,micro_f1,precision_macro,recall_macro,macro_f1,exact_match,hamming_loss,threshold,training_seconds
0,b1,phone,0.540404,0.286096,0.374126,0.060011,0.100894,0.075246,0.219731,0.089188,0.25,2190.590744
1,b1,laptop,0.326531,0.618357,0.427379,0.161164,0.265578,0.179640,0.017143,0.093333,0.15,3377.294026
2,b2,overall,0.668224,0.544416,0.600000,0.206778,0.208288,0.200418,0.425829,0.036972,0.20,11376.997171
3,b2,phone,0.661238,0.542781,0.596182,0.156392,0.164958,0.152674,0.358744,0.045673,0.20,11376.997171
4,b2,laptop,0.674627,0.545894,0.603471,0.195582,0.192403,0.187221,0.468571,0.031429,0.20,11376.997171


,domain,b1_micro_f1,b2_micro_f1,delta_micro_f1_b2_minus_b1,b1_macro_f1,b2_macro_f1,delta_macro_f1_b2_minus_b1
0,phone,0.374126,0.596182,0.222056,0.075246,0.152674,0.077428
1,laptop,0.427379,0.603471,0.176092,0.179640,0.187221,0.007580


In [15]:

metrics_df.to_csv(RESULTS_DIR / "baseline_metrics.csv", index=False)
comparison_df.to_csv(RESULTS_DIR / "baseline_comparison.csv", index=False)
per_label_df.to_csv(RESULTS_DIR / "baseline_per_label_metrics.csv", index=False)
history_df.to_csv(RESULTS_DIR / "baseline_training_history.csv", index=False)
predictions_df.to_csv(RESULTS_DIR / "baseline_test_predictions.csv", index=False)

with open(RESULTS_DIR / "baseline_run_config.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "model_name": MODEL_NAME,
            "seed": SEED,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "smoke_test": SMOKE_TEST,
            "save_model_weights": SAVE_MODEL_WEIGHTS,
            "common_labels": COMMON,
            "phone_specific_labels": PHONE_SPECIFIC,
            "laptop_specific_labels": LAPTOP_SPECIFIC,
            "device": str(DEVICE),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

required_outputs = [
    RESULTS_DIR / "baseline_metrics.csv",
    RESULTS_DIR / "baseline_comparison.csv",
    RESULTS_DIR / "baseline_per_label_metrics.csv",
    RESULTS_DIR / "baseline_training_history.csv",
    RESULTS_DIR / "baseline_test_predictions.csv",
    RESULTS_DIR / "baseline_run_config.json",
]

checks = {
    "b1_phone_metrics": ((metrics_df["model"] == "b1") & (metrics_df["segment"] == "phone")).any(),
    "b1_laptop_metrics": ((metrics_df["model"] == "b1") & (metrics_df["segment"] == "laptop")).any(),
    "b2_overall_metrics": ((metrics_df["model"] == "b2") & (metrics_df["segment"] == "overall")).any(),
    "valid_micro_f1": metrics_df["micro_f1"].between(0, 1).all(),
    "valid_macro_f1": metrics_df["macro_f1"].between(0, 1).all(),
    "outputs_saved": all(path.exists() for path in required_outputs),
}

validation_df = pd.DataFrame([{"check": k, "passed": bool(v)} for k, v in checks.items()])
display(validation_df)

if not validation_df["passed"].all():
    failed = validation_df.loc[~validation_df["passed"], "check"].tolist()
    raise ValueError(f"BASELINE VALIDATION FAILED: {failed}")

print("C1 B1 + B2 BASELINES: PASSED")


,check,passed
0,b1_phone_metrics,True
1,b1_laptop_metrics,True
2,b2_overall_metrics,True
3,valid_micro_f1,True
4,valid_macro_f1,True
5,outputs_saved,True


C1 B1 + B2 BASELINES: PASSED
